# 02 Data Cleaning

This notebook creates the first clean modeling dataset for the credit risk project.

## 1. Import Libraries

In [ ]:
# Import libraries for data loading and analysis
import pandas as pd
import numpy as np

## 2. Load Raw Data

In [ ]:
# Set the path to the raw LendingClub CSV file
data_path = "../data/raw/raw_data.csv"

# Load the raw dataset
df = pd.read_csv(data_path, low_memory=False)

# Store the number of rows before filtering
rows_before_filtering = len(df)

print(f"Rows before filtering: {rows_before_filtering:,}")

## 3. Create Default Flag

In [ ]:
# Map loan_status values to the target variable
# Fully Paid means the borrower did not default
# Charged Off and Default mean the borrower defaulted
default_flag_map = {
    "Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1
}

# Create default_flag using the mapping above
# Any loan_status not in the mapping will become missing for now
df["default_flag"] = df["loan_status"].map(default_flag_map)

### Milestone 1 Analysis: Target Mapping

**Observation**  
`default_flag` maps clear non-default outcomes to `0` and clear default outcomes to `1`.

**Interpretation**  
This creates a binary target suitable for a first Probability of Default model. Other loan statuses are intentionally left unmapped so they can be removed from version 1.

**Modeling Implication**  
A transparent target definition is one of the most important controls in credit risk modeling. It keeps the model aligned with the business question: whether a borrower defaults.

**Decision**  
Use this binary `default_flag` as the target for the first PD model.

## 4. Filter to Modeling Rows

In [ ]:
# Keep only rows with loan_status values that are useful for the first model version
modeling_df = df[df["default_flag"].notna()].copy()

# Convert default_flag from float to integer after filtering
modeling_df["default_flag"] = modeling_df["default_flag"].astype(int)

# Store the number of rows after filtering
rows_after_filtering = len(modeling_df)

print(f"Rows before filtering: {rows_before_filtering:,}")
print(f"Rows after filtering: {rows_after_filtering:,}")

### Milestone 1 Analysis: Filtering to Known Outcomes

**Observation**  
Rows are filtered to keep only loans with a clear mapped value for `default_flag`.

**Interpretation**  
This removes statuses that are not part of the first target definition, such as loans that may still be current or otherwise unresolved.

**Modeling Implication**  
Training on unresolved outcomes can weaken the target and confuse model learning. Filtering helps create a cleaner supervised learning dataset.

**Decision**  
Use the filtered dataset for version 1 and document any excluded statuses if the target definition changes later.

## 5. Check Default Flag

In [ ]:
# Show the number of non-default and default loans
print("Default flag value counts:")
print(modeling_df["default_flag"].value_counts().sort_index())

## 6. Calculate Default Rate

In [ ]:
# The default rate is the average of default_flag because 1 means default and 0 means non-default
default_rate = modeling_df["default_flag"].mean()

print(f"Default rate: {default_rate:.2%}")

### Milestone 1 Analysis: Default Rate and Class Balance

**Observation**  
The default rate gives the share of loans where `default_flag = 1`.

**Interpretation**  
This is the baseline risk level in the modeling sample. It also shows whether the target is imbalanced.

**Modeling Implication**  
If defaults are much less common than non-defaults, model evaluation should use metrics beyond accuracy, such as ROC AUC, precision-recall, recall, and calibration.

**Decision**  
Track the default rate as a baseline benchmark for the first PD model.

## 7. Clean Selected Columns

In [ ]:
# Start from the filtered modeling dataset
loans_clean = modeling_df.copy()

# Remove the percent sign from int_rate and convert it to a number
loans_clean["int_rate"] = (
    loans_clean["int_rate"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)
loans_clean["int_rate"] = pd.to_numeric(loans_clean["int_rate"], errors="coerce")

# Remove the percent sign from revol_util and convert it to a number
loans_clean["revol_util"] = (
    loans_clean["revol_util"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
)
loans_clean["revol_util"] = pd.to_numeric(loans_clean["revol_util"], errors="coerce")

# Extract the number of months from term and convert it to an integer
loans_clean["term"] = loans_clean["term"].astype(str).str.extract("(\\d+)")
loans_clean["term"] = pd.to_numeric(loans_clean["term"], errors="coerce").astype("Int64")

# Convert employment length text into numeric years
emp_length_map = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10
}
loans_clean["emp_length"] = loans_clean["emp_length"].map(emp_length_map)

# Check the cleaned columns
loans_clean[["int_rate", "revol_util", "term", "emp_length"]].head()

### Milestone 1 Analysis: Cleaned Numeric Fields

**Observation**  
`int_rate`, `revol_util`, `term`, and `emp_length` are converted from text-like fields into numeric values.

**Interpretation**  
These variables are likely to be useful risk drivers because they describe pricing, utilization, loan duration, and employment history.

**Modeling Implication**  
Numeric conversion makes these fields usable in machine learning pipelines and allows missing values or outliers to be handled consistently.

**Decision**  
Keep these cleaned fields in `loans_clean` for future feature selection and modeling.

## 8. Remove Leakage Columns

In [ ]:
# Leakage columns are variables that would only be known after the loan outcome is observed.
# Including them in a Probability of Default model would make the model unrealistically accurate.
# For example, payment, recovery, hardship, and settlement fields can reveal what happened after origination.
leakage_columns = [
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "settlement_status",
    "hardship_flag",
    "debt_settlement_flag"
]

# Build the first candidate feature list for the Probability of Default model.
# This excludes leakage columns and also excludes default_flag, which is the target variable.
pd_model_candidate_columns = [
    column
    for column in loans_clean.columns
    if column not in leakage_columns and column != "default_flag"
]

print(f"Number of leakage columns listed: {len(leakage_columns)}")
print(f"Number of PD model candidate columns: {len(pd_model_candidate_columns)}")

### Milestone 1 Analysis: Data Leakage Control

**Observation**  
Known leakage columns are listed separately and excluded from `pd_model_candidate_columns` along with the target variable.

**Interpretation**  
Payment, recovery, hardship, and settlement variables can reveal information that occurs after the borrower has already performed or defaulted.

**Modeling Implication**  
Including these fields would make the PD model look stronger in testing than it would be in a real lending decision. This is one of the highest-risk mistakes in credit risk modeling.

**Decision**  
Do not use leakage columns in the first PD model. Continue reviewing candidate features for any additional post-outcome or unavailable-at-origination variables.

## 9. Save Clean Modeling Dataset

In [ ]:
# Save the clean modeling dataset for the next project step
output_path = "../data/processed/loans_clean_v1.csv"
loans_clean.to_csv(output_path, index=False)

print(f"Clean modeling dataset saved to: {output_path}")

### Milestone 1 Analysis: Clean Dataset Artifact

**Observation**  
The cleaned version 1 dataset is saved as `loans_clean_v1.csv`.

**Interpretation**  
This creates a stable handoff point between data cleaning, EDA, and future modeling work.

**Modeling Implication**  
Using a saved artifact helps keep experiments reproducible and prevents each notebook from redefining the sample differently.

**Decision**  
Use `loans_clean_v1.csv` as the source dataset for Milestone 1 EDA and the first modeling notebook.